In [ ]:
import camb,os,struct
import numpy as np
import matplotlib.pyplot as plt

# 读取参数
# Universe = '_2400_6144' 
Path = '/Users/chenbinghang/Library/CloudStorage/OneDrive-个人/陈冰航/天文/sim_2D/test1/'
z = np.loadtxt(Path+'/code/z_checkpoint.txt');n_checkpoint = len(z);a = 1. / (1. + z);num_step = np.zeros(n_checkpoint)
Redshift = '200.000'

plt.rcParams.update({
    'figure.figsize': (12, 8),
    'figure.facecolor': 'w',
    'figure.dpi': 300,
    'lines.linewidth': 1.0,
    'axes.spines.top': True,  # 显示顶部边框线
    'axes.spines.right': True,  # 显示右边边框线
    'xtick.bottom': True,  # 显示x轴刻度线
    'xtick.direction': 'in',  # x轴刻度线朝内
    'ytick.left': True,  # 显示y轴刻度线
    'ytick.direction': 'in',  # y轴刻度线朝内
    'xtick.top': True,  # 显示x轴刻度线（顶部）
    'ytick.right': True,# 显示y轴刻度线（右侧）
    'axes.linewidth': 1.0,
    'axes.xmargin': 0.03,
    'axes.ymargin': 0.03,
    'axes.spines.right': False,
    'axes.spines.top': False,
    'axes.grid': False,
    'axes.grid.which': 'both',
    # 'axes.grid.alpha': 0.5,
    'axes.labelpad': 8.0,
    'axes.labelsize': 10,
    'axes.labelcolor': 'k',
    'axes.axisbelow': True,
    'xtick.minor.visible': True,
    'ytick.minor.visible': True,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.size': 4,
    'ytick.major.size': 4,
    'xtick.minor.size': 2,
    'ytick.minor.size': 2,
    'xtick.major.width': 1.0,
    'ytick.major.width': 1.0,
    'xtick.minor.width': 1.0,
    'ytick.minor.width': 1.0,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'font.size': 10,
    'text.usetex': False,
    'legend.fontsize': 12,
    'legend.frameon': False,
    'lines.markersize': 6,
    'axes.prop_cycle': plt.cycler('color', [[0, 0, 0], [1, 0.2, 0.2], [0.4, 0.7, 1], [0.2, 0.8, 0.3], [1, 0.7, 0.3], [0.6, 0.3, 1], [1, 0.7, 0.7], [0.5, 0.5, 0.5]])
})

def get_sim_info(prefix):
    sim = {}

    with open(prefix + 'info.bin', 'rb') as fid:
        sim['np'] = struct.unpack('q', fid.read(8))[0]

        sim['izipx'] = struct.unpack('q', fid.read(8))[0]
        sim['izipv'] = struct.unpack('q', fid.read(8))[0]
        
        sim['nnt'] = struct.unpack('q', fid.read(8))[0]
        sim['nt'] = struct.unpack('q', fid.read(8))[0]
        sim['ncell'] = struct.unpack('q', fid.read(8))[0]
        sim['ncb'] = struct.unpack('q', fid.read(8))[0]

        sim['istep'] = struct.unpack('q', fid.read(8))[0]
        sim['cur_checkpoint'] = struct.unpack('q', fid.read(8))[0]
        # sim['cur_halofind'] = struct.unpack('q', fid.read(8))[0]
        sim['cur_powerpoint'] = struct.unpack('q', fid.read(8))[0]


        sim['calculate_PK'] = struct.unpack('q', fid.read(8))[0]
        sim['cic_iapm'] = struct.unpack('q', fid.read(8))[0]

        sim['a'] = struct.unpack('f', fid.read(4))[0]
        sim['t'] = struct.unpack('f', fid.read(4))[0]
        sim['tau'] = struct.unpack('f', fid.read(4))[0]
        sim['dt'] = struct.unpack('4f', fid.read(16))
        sim['mass_p'] = struct.unpack('f', fid.read(4))[0]
        sim['m_nu'] = struct.unpack('3f', fid.read(12))
        sim['Mass_nu'] = struct.unpack('f', fid.read(4))[0]
        sim['box'] = struct.unpack('f', fid.read(4))[0]

        sim['h0'] = struct.unpack('f', fid.read(4))[0]
        sim['omega_m'] = struct.unpack('f', fid.read(4))[0]
        sim['omega_l'] = struct.unpack('f', fid.read(4))[0]
        sim['s8'] = struct.unpack('f', fid.read(4))[0]
        sim['vsim2phys'] = struct.unpack('f', fid.read(4))[0]
        sim['sigma_vres'] = struct.unpack('f', fid.read(4))[0]
        sim['sigma_vi'] = struct.unpack('f', fid.read(4))[0]
        sim['z_i'] = struct.unpack('f', fid.read(4))[0]
        sim['vz_max'] = struct.unpack('f', fid.read(4))[0]

    sim['nc'] = sim['nt'] * sim['nnt']
    sim['ng_global'] = sim['nnt'] * sim['nt'] * 4
    G = 6.67408e-11  # in m^3 kg^-1 s^-2
    rho_crit = 30000 / 8 / 3.14159 / G / (3.086e19) ** 2  # in kg m^-3 h^-2
    rho_crit = rho_crit / 1.98855e30 * (3.086e22) ** 3  # in M_solar Mpc^-3 h^-2
    sim['mass_p_solar'] = rho_crit * sim['omega_m'] * sim['box'] ** 3 / sim['np'] / sim['h0']
    sim['mass_p_solar_per_h'] = rho_crit * sim['omega_m'] * sim['box'] ** 3 / sim['np']

    return sim

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']

sim = get_sim_info(Path + Redshift + '_')
os.mkdir(Path+'/fig/') if not os.path.exists(Path+'/fig/') else None
    
ng = sim['ng_global']

# 输出参数
print('ng_global =  {:d}\nbox       =  {:.1f}\nistep     =  {:d}\nmass_p    =  {:.1f}'.\
      format(sim['ng_global'],sim['box'],sim['istep'],sim['mass_p_solar']))

In [4]:
def loadfield2d(fn):
    """
    fn: filename
    Returns:
    a: 2D field
    """
    fid = open(fn, 'rb')
    # fid = open(fn, 'rb', 'big')  # big endian
    p1 = np.fromfile(fid, dtype=np.float32)  # real*4
    fid.close()
    n = round(len(p1) ** (1/2))
    if (n**2 != len(p1)):
        print('shape no mach')
        print(n,n**2,len(p1))
        return 0,0
    a = np.reshape(p1, (n, n),order='F')
    # print(fn,n)
    return a,n

def plt_delta(name,box='400',Redshift='5.000',cl=[-5,5],cmap='bwr',save=True,times=1):
    delta,n = loadfield2d(f'{Path}{Redshift}_{name}.bin')
    plt.figure(figsize=(4,3),dpi=300)
    plt.imshow(delta*times, interpolation='none',cmap=cmap)
    # plt.imsave(f'200_{name}.png', delta.T, cmap=cmap, origin='lower')
    plt.title(f'{box}_{name}_{Redshift}')
    if (cl != 'auto'):
        plt.clim(cl[0],cl[1])
    # plt.clim(-1,3)
    if (times == 1):
        plt.colorbar(label=f'{name}')
    else:
        plt.colorbar(label=f'{times:.1e} {name}')
    print(np.max(delta),np.min(delta))
    if save:
        os.makedirs(f'{Path}/fig', exist_ok=True)
        plt.savefig(f'{Path}/fig/{box}_{name}_{Redshift}.pdf', bbox_inches='tight')
        plt.close()
    else:
        plt.show()
def loadpower(filename):
    n_row_xi = 10
    fid = open(filename, 'rb')  # 以二进制模式打开文件

    xi = np.fromfile(fid, dtype='float32')  # 读取数据到一维数组
    fid.close()

    xi = np.reshape(xi, (int(len(xi) / n_row_xi), n_row_xi))  # 重新形状为二维数组
    ksim = xi[:, 1]  # 提取第二列作为 ksim

    return ksim, xi

def load2dxpos(fn):
    fid = open(fn, 'rb')
    p1 = np.fromfile(fid, dtype=np.float32)
    fid.close()
    n = round(len(p1)/2)
    if (n*2 != len(p1)):
        print('shape no mach')
        print(n,n**2,len(p1))
        return 0,0
    a = np.reshape(p1, (2, n),order='F')
    # print(fn,n)
    return a,int(np.sqrt(n))

In [20]:
Path = '/Users/chenbinghang/Library/CloudStorage/OneDrive-个人/陈冰航/天文/sim_2D/3000_3072/'

In [ ]:
# plt ic
import re


def match_para(para):
    # 读取Fortran文件
    file_path = './parameters.f90'
    with open(file_path, 'r') as file:
        content = file.read()

    # 使用正则表达式匹配模式
    pattern = r'parameter\s*::\s*%s\s*=\s*([^\s]+)'%para
    match = re.search(pattern, content)

    if match:
        variable_value = match.group(1).strip()
        return float(variable_value)
    else:
        print(para,"Pattern not found in the file.")
        sys.exit()

# os.system('source ./Apple_silcon.sh && cd utilities && make && cd .. && ./utilities/ic.x && ./utilities/density_power.x')
end=400
Redshift = '200.000'
title = 'box = %d'%match_para('box')

pos,n = load2dxpos(f'{Path}{Redshift}_xp.bin')
# 加载功率谱
def loadpower(filename):
    n_row_xi = 10
    fid = open(filename, 'rb')  # 以二进制模式打开文件

    xi = np.fromfile(fid, dtype='float32')  # 读取数据到一维数组
    fid.close()

    xi = np.reshape(xi, (int(len(xi) / n_row_xi), n_row_xi))  # 重新形状为二维数组
    ksim = xi[:, 1]  # 提取第二列作为 ksim

    return ksim, xi
def power_error(k,pk,k_camb,pk_camb):
    from scipy.stats import gamma
    from scipy.interpolate import interp1d
    
    n=np.array(pk[:,0])
    lower = gamma.ppf(0.1587, a=n, scale=1/n)
    upper = gamma.ppf(0.8413, a=n, scale=1/n)

    interp_camb = interp1d(k_camb,pk_camb, kind='linear', bounds_error=False, fill_value="extrapolate")
    pk_i = interp_camb(k)

    return pk_i,pk_i*upper,pk_i*lower
# cam = np.loadtxt('tf_wmap9/camb_82031666_matterpower_z0.dat.txt')
k, xi = loadpower(f'{Path}' + Redshift + '_power.bin')
[k_camb,pk_camb] = [np.loadtxt(f'{Path}neutrinos/IC/Pcb_ic.txt')[:,0],np.loadtxt(f'{Path}neutrinos/IC/Pcb_ic.txt')[:,1]]

if (1):
    plt.figure(figsize=(16,4),dpi=300)
    plt.subplot(1,4,2)
    plt.loglog(k[:-end], xi[:-end, 2],label='$Pk_{1200_\_ 6144}$',color=colors[2], linewidth=2)
    plt.loglog(k_camb,pk_camb,'--',label='$Pk_{CAMB}$',color=colors[0], linewidth=2)
    pk,pk_u,pk_l = power_error(k,xi,k_camb,pk_camb)
    plt.fill_between(k[:-end], pk_u[:-end],pk_l[:-end],color=colors[2], alpha=0.3,label='Cosmic variance')
    # plt.xlim(5e-3,2e2)
    # plt.ylim(1e-5,4e-2)
    plt.xlim(np.nanmin(k[:-end])/1.2,np.nanmax(k[:-end])*1.2)
    plt.title(title)
    plt.grid(True,'both')

    # 绘制pos的散点图
    plt.subplot(1,4,1)
    plt.scatter(pos[0,:], pos[1,:], c='black', s=.4, marker='o', edgecolors='none',alpha=0.5)
    # xy轴一样长
    plt.axis('equal')
    plt.xlim(-1,n+1)
    plt.ylim(-1,n+1)
    #关闭坐标轴
    plt.axis('off')
    # plt.title(title,fontsize=4)
    plt.subplot(1,4,3)
    delta,n = loadfield2d(f'{Path}{Redshift}_delta_L.bin')
    plt.imshow(delta,cmap='gray', origin='lower')
    plt.title('$\delta_L$')
    # plt.clim(-1,3)
    plt.colorbar()
    plt.axis('equal')
    plt.axis('off')

    plt.subplot(1,4,4)
    delta,n = loadfield2d(f'{Path}{Redshift}_phi1.bin')
    plt.imshow(delta,cmap='gray', origin='lower')
    plt.title('$\phi_L$')
    # plt.clim(-1000,1000)
    plt.colorbar()
    plt.axis('equal')
    plt.axis('off')

In [ ]:

[k_camb,pk_camb] = [np.loadtxt(f'{Path}neutrinos/IC/Pcb_ic.txt')[:,0],np.loadtxt(f'{Path}neutrinos/IC/Pcb_ic.txt')[:,1]]
for Redshift in ['2.000','3.000','4.000','5.000']:
    k, xi = loadpower(f'{Path}' + Redshift + '_power.bin')
    plt.loglog(k[:-end], xi[:-end, 2],label='$Pk_{1200_\_ 6144}$',color=colors[2], linewidth=2)
    plt.loglog(k_camb,pk_camb,'--',label='$Pk_{CAMB}$',color=colors[0], linewidth=2)
    pk,pk_u,pk_l = power_error(k,xi,k_camb,pk_camb)
    plt.fill_between(k[:-end], pk_u[:-end],pk_l[:-end],color=colors[2], alpha=0.3,label='Cosmic variance')
    # plt.xlim(5e-3,2e2)
    # plt.ylim(1e-5,4e-2)
    plt.xlim(np.nanmin(k[:-end])/1.2,np.nanmax(k[:-end])*1.2)
    plt.title(title)
    plt.grid(True,'both')

In [ ]:
for Redshift in ['2.000','3.000','4.000','5.000']:
    plt_delta('delta_c',box='%d'%match_para('box'),Redshift=Redshift,cl=[-1,3],cmap='gray',save=False,times=1)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
def loadpower(filename):
    n_row_xi = 10
    fid = open(filename, 'rb')  # 以二进制模式打开文件

    xi = np.fromfile(fid, dtype='float32')  # 读取数据到一维数组
    fid.close()

    xi = np.reshape(xi, (int(len(xi) / n_row_xi), n_row_xi))  # 重新形状为二维数组
    ksim = xi[:, 1]  # 提取第二列作为 ksim

    return ksim, xi
# for box in[2000,1000,500,400]:
#     Path = f'/Users/chenbinghang/Library/CloudStorage/OneDrive-个人/陈冰航/天文/sim_2D/{box:d}_1024/'
plt.figure(figsize=(4 ,3),dpi=300)
for Redshift in ['2.000','3.000','4.000','5.000']:
# for Redshift in ['0.000','0.100','0.500','1.000','2.000','3.000','4.000','5.000']:
    k, xiLN = loadpower(f'{Path}' + Redshift + '_Cpower_LN.bin')

    r_LN  = xiLN[:,8]/np.sqrt(xiLN[:,5]*xiLN[:,2])

    plt.plot(k, r_LN, label=r'$r_{LN},z=$'+Redshift)
plt.axhline(y=0.8, color='k', linestyle='--')
plt.axvline(x=0.1, color='k', linestyle='--')
plt.axhline(y=0.5, color='b', linestyle='--')
plt.axvline(x=0.2, color='b', linestyle='--')
plt.xscale('log')
plt.legend(loc='upper right')
plt.ylim(-0.01,1.1)
plt.xlim(0.004,40)
plt.xlabel(r'$k$ [$h$/Mpc]')
plt.show()


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
def loadpower(filename):
    n_row_xi = 10
    fid = open(filename, 'rb')  # 以二进制模式打开文件

    xi = np.fromfile(fid, dtype='float32')  # 读取数据到一维数组
    fid.close()

    xi = np.reshape(xi, (int(len(xi) / n_row_xi), n_row_xi))  # 重新形状为二维数组
    ksim = xi[:, 1]  # 提取第二列作为 ksim

    return ksim, xi
# for box in[2000,1000,500,400]:
#     Path = f'/Users/chenbinghang/Library/CloudStorage/OneDrive-个人/陈冰航/天文/sim_2D/{box:d}_1024/'
if (1) :
    box = 100
    Path = f'/Users/chenbinghang/Library/CloudStorage/OneDrive-个人/陈冰航/天文/sim_2D/100_128/'
    plt.figure(figsize=(4 ,3),dpi=300)
    for Redshift in ['2.000','3.000','4.000','5.000']:
    # for Redshift in ['0.000','0.100','0.500','1.000','2.000','3.000','4.000','5.000']:
        k, xiLN = loadpower(f'{Path}' + Redshift + '_Cpower_LN.bin')

        r_LN  = xiLN[:,8]/np.sqrt(xiLN[:,5]*xiLN[:,2])

        plt.plot(k, r_LN, label=r'$r_{LN},z=$'+Redshift)
    plt.axhline(y=0.8, color='k', linestyle='--')
    plt.axvline(x=0.1, color='k', linestyle='--')
    plt.axhline(y=0.5, color='b', linestyle='--')
    plt.axvline(x=0.2, color='b', linestyle='--')
    plt.xscale('log')
    plt.legend(loc='upper right')
    plt.ylim(-0.01,1.1)
    plt.xlim(0.004,40)
    plt.xlabel(r'$k$ [$h$/Mpc]')
    plt.title('$L_{box} = %d$'%box)
    plt.show()


In [ ]:
conda install Treecorr